### Build gold dimension table

This notebook builds the Gold dimension gold_company_profile by enriching company master data with the latest PEO compensation, latest NEO compensation, and latest reported net income extracted from the Silver DEF 14A table.
It joins these “latest” metrics to the company table and adds a flag indicating whether the company has already reported current‑year (2025) PEO compensation.
The result is a clean, analytics‑ready Gold table used for Power BI dashboards and executive compensation reporting.

In [1]:
from pyspark.sql import functions as F

# Load Silver tables
company = spark.table("silver_company_info")
def14a = spark.table("sec_def14a_silver")

# Latest PEO
latest_peo = (
    def14a
    .filter(F.col("is_latest_peo") == 1)
    .select(
        "cik",
        F.col("peo_total_comp").alias("most_current_peo_total_comp"),
        F.col("fiscal_year").alias("most_current_peo_fiscal_year")
    )
)

# Latest NEO
latest_neo = (
    def14a
    .filter(F.col("is_latest_neo") == 1)
    .select(
        "cik",
        F.col("neo_avg_total_comp").alias("most_current_neo_total_comp"),
        F.col("fiscal_year").alias("most_current_neo_fiscal_year")
    )
)

# Latest Net Income
latest_net = (
    def14a
    .filter(F.col("is_latest_net_income") == 1)
    .select(
        "cik",
        F.col("net_income").alias("most_current_net_income")
    )
)

# Build Gold Dimension
gold_company_profile = (
    company
    .join(latest_peo, "cik", "left")
    .join(latest_neo, "cik", "left")
    .join(latest_net, "cik", "left")
    .withColumn(
        "has_current_year_flag",
        F.when(F.col("most_current_peo_fiscal_year") == 2025, 1).otherwise(0)
    )
    .drop("ingestion_ts")
)

# Write Gold table
gold_company_profile.write.format("delta").mode("overwrite").saveAsTable("gold_company_profile")

print("Gold Company Profile created.")


StatementMeta(, 41223958-047e-46b1-8127-41d55ce15ae7, 3, Finished, Available, Finished, False)

Gold Company Profile created.
